<a href="https://colab.research.google.com/github/yg36/CNN-on-Sign-Language-/blob/main/CNN_on_Sign_Language_MINIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader

import pandas as pd

In [2]:
import kagglehub

path = kagglehub.dataset_download("datamunge/sign-language-mnist")

Using Colab cache for faster access to the 'sign-language-mnist' dataset.


In [3]:
training_df = pd.read_csv(f"{path}/sign_mnist_train.csv")
validation_df = pd.read_csv(f"{path}/sign_mnist_test.csv")

In [4]:
IMG_HEIGHT = 28
IMG_WIDTH = 28
IMG_CHS = 1

In [5]:
labels = sorted(training_df["label"].unique())
label_map = {label:idx for idx, label in enumerate(labels)}

training_df["label"] = training_df["label"].map(label_map)
validation_df["label"] = validation_df["label"].map(label_map)

In [6]:
class SignDataset(Dataset):
  def __init__(self, base_df):
    x_df = base_df.copy()
    y_df = x_df.pop("label")
    x_df = x_df.values / 255    #normalization
    x_df = x_df.reshape(-1, IMG_CHS, IMG_HEIGHT, IMG_WIDTH)
    self.xs = torch.tensor(x_df).float()
    self.ys = torch.tensor(y_df)

  def __getitem__(self,idx):
    x = self.xs[idx]
    y = self.ys[idx]
    return x,y

  def __len__(self):
    return len(self.xs)

In [7]:
training_data = SignDataset(training_df)
training_loader = DataLoader(training_data, batch_size=64, shuffle=True)
training_N = len(training_loader.dataset)

validation_data = SignDataset(validation_df)
validation_loader = DataLoader(validation_data, batch_size=64, shuffle=True)
validation_N = len(validation_loader.dataset)

In [8]:
N_CLASSES = 24
KERNEL_SIZE = 3
FLATTENED_IMG_SIZE = 75*3*3

model = nn.Sequential(
    # First Convulation
    nn.Conv2d(IMG_CHS, 25, KERNEL_SIZE, stride = 1, padding = 1),
    nn.BatchNorm2d(25),
    nn.ReLU(),
    nn.MaxPool2d(2,stride=2),

    nn.Conv2d(25,50, KERNEL_SIZE, stride=1, padding=1),
    nn.BatchNorm2d(50),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.MaxPool2d(2, stride=2),

    nn.Conv2d(50,75, KERNEL_SIZE, stride=1, padding=1),
    nn.BatchNorm2d(75),
    nn.ReLU(),
    nn.MaxPool2d(2, stride=2),

    nn.Flatten(),
    nn.Linear(FLATTENED_IMG_SIZE, 512),
    nn.Dropout(0.3),
    nn.ReLU(),
    nn.Linear(512, N_CLASSES)
)

In [9]:
# model = torch.compile()
loss_function = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=0.001)

In [10]:
def get_batch_accuracy(output, y, N):
  pred = output.argmax(dim=1, keepdim=True)
  correct = pred.eq(y.view_as(pred)).sum().item()
  return correct/N

In [11]:
def train():
  loss=0
  accuracy = 0

  model.train()
  for x,y in training_loader:
    output = model(x)
    optimizer.zero_grad()
    batch_loss = loss_function(output,y)
    batch_loss.backward()
    optimizer.step()

    loss+=batch_loss.item()
    accuracy +=get_batch_accuracy(output, y, training_N)

  print("Training Loss:", loss)
  print("Training Accuracy: ", accuracy)


In [12]:
def validate():
  loss=0
  accuracy=0

  model.eval()
  with torch.no_grad():
    for x,y in validation_loader:
      output = model(x)

      loss+= loss_function(output, y).item()
      accuracy += get_batch_accuracy(output, y, validation_N)

    print("Validation Loss: ", loss)
    print("Validation Accuracy: ", accuracy)

In [13]:
%%time
EPOCHS = 20
for epoch in range(EPOCHS):
  print("Epoch: ", epoch)
  train()
  validate()

Epoch:  0
Training Loss: 164.1811613915488
Training Accuracy:  0.8912402112547753
Validation Loss:  18.098072350025177
Validation Accuracy:  0.9538482989403211
Epoch:  1
Training Loss: 5.6393942683935165
Training Accuracy:  0.9978510289564677
Validation Loss:  14.367036234471016
Validation Accuracy:  0.9569157836028978
Epoch:  2
Training Loss: 4.76685659569921
Training Accuracy:  0.9977781824804154
Validation Loss:  31.691567581146955
Validation Accuracy:  0.9295872838817603
Epoch:  3
Training Loss: 3.747225699160481
Training Accuracy:  0.9977053360043646
Validation Loss:  8.867705324068083
Validation Accuracy:  0.9656999442275491
Epoch:  4
Training Loss: 6.400218662045518
Training Accuracy:  0.9958113276270193
Validation Loss:  11.766059423447587
Validation Accuracy:  0.9652816508644705
Epoch:  5
Training Loss: 0.28457495083603135
Training Accuracy:  0.9999271535239413
Validation Loss:  9.863835824537091
Validation Accuracy:  0.9676519799219161
Epoch:  6
Training Loss: 0.0941372302668